# Trajectum vs STAR: Gamma Fit Overlay (0-5% centrality)

Overlays the Trajectum event-by-event mean-$p_T$ distribution and its
Gamma fit against the STAR BES-I published Gamma curve (same $\alpha$,
$\beta$ parameterization), for direct visual comparison. This is
trimmed down to just that one comparison -- no separate bar-chart
comparisons, no second STAR table.

In [ ]:
import os
import re
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import gamma


In [ ]:
# ------------------------------------------------------------------
# 1. Files to scan.
#
# Trajectum pT-study output, one file per energy, 200 MeV track cut.
# 11.5 GeV is newly added to match the STAR HEPData point at the same
# energy -- update the path below once that run/export exists.
# ------------------------------------------------------------------
filepaths = [
    "src/monotonic_ptfluc_study/7.7GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/11.5GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/19GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/27GeV_200MeVcut_ptfluc_study.h5",
    "src/monotonic_ptfluc_study/200GeV_200MeVcut_ptfluc_study.h5",
]

# Trajectum nominal energy -> matching STAR sqrt(s_NN) label in the
# HEPData table. Only these energies have a STAR 0-5% point to compare
# against.
STAR_ENERGY_MATCH = {
    7.7: 7.7,
    11.5: 11.5,
    19.0: 19.6,
    27.0: 27.0,
    200.0: 200.0,
}

# Published STAR BES-I alpha values (Au+Au, 0-5% central), from the
# Gamma-fit parameterization table (same alpha/beta fit form used
# here). Keyed by sqrt(s_NN) [GeV].
STAR_ALPHA_BESI = {
    7.7: 789.5,
    11.5: 930.9,
    19.6: 1089.1,
    27.0: 1166.3,
    39.0: 1241.8,
    62.4: 1271.5,
    200.0: 1465.3,
}

# Published STAR BES-I beta values [GeV^-1] (Au+Au, 0-5% central), same
# table as alpha above. NOTE: despite the GeV^-1 label, this lines up
# with our fit's `scale` parameter directly (mean ~= a * scale for
# loc ~= 0), not 1/scale -- verified against expected <pT> ~0.55-0.65 GeV,
# and now independently CONFIRMED by the second STAR table below, which
# explicitly states beta = scale in its fit-form definition.
STAR_BETA_BESI = {
    7.7: 0.00070137,
    11.5: 0.00058381,
    19.6: 0.00049720,
    27.0: 0.00046754,
    39.0: 0.00044513,
    62.4: 0.00044472,
    200.0: 0.00040798,
}

TARGET_CENT_LABEL = "0-5%"


In [ ]:
# ------------------------------------------------------------------
# 2. Helpers: gamma fit to the event-by-event mean-pT distribution
#    (same method as eventbyeventmeanptgammafit.ipynb).
# ------------------------------------------------------------------
def _energy_from_filename(path):
    m = re.search(r"(\d+(?:\.\d+)?)GeV", str(path))
    if not m:
        raise ValueError(f"Couldn't parse energy from filename: {path}")
    return float(m.group(1))


def _detect_track_group(hdf, base="eventbyeventmeanptcharged"):
    """Return the single track-cut group name under `base`
    (e.g. "STARTPC", "STARTPC150", "STARTPC200MeV"). Raises if the
    file contains more than one, in which case pass track_group
    explicitly to get_gamma_stats_from_file."""
    groups = list(hdf[base].keys())
    if len(groups) != 1:
        raise ValueError(
            f"Expected exactly one track-cut group under '{base}', "
            f"found {groups}. Pass track_group explicitly."
        )
    return groups[0]


def gamma_func(x, A, a, scale):
    """Scaled gamma PDF used as the fit model, loc fixed at 0 to match
    STAR's alpha/beta parameterization (f(x; a, beta) = x^(a-1) e^(-x/beta)
    / (Gamma(a) beta^a)), i.e. gamma.pdf(x, a, loc=0, scale=beta)."""
    return A * gamma.pdf(x, a, loc=0, scale=scale)


def fit_gamma_to_bin(binc, y):
    """Fit a scaled gamma PDF (loc=0) to one centrality bin's
    distribution. Returns (popt, pcov).

    Fits in log-space (log(y) vs log(model)) rather than linear space.
    These distributions span several decades from peak to tail (~30
    down to ~1e-4), and an unweighted linear-space fit lets the peak's
    large absolute residuals completely dominate the sum of squares,
    leaving the tail systematically underfit. Log-space residuals
    weight a given relative/fractional miss the same whether it's at
    the peak or in the tail."""
    mask = y > 1e-6  # ignore near-zero/noise bins for the fit
    mean_guess = binc[np.argmax(y)]

    p0 = [1.0, 100, mean_guess / 100]
    bounds = (
        [0, 1, 1e-5],           # lower bounds: A, a, scale
        [1000, 10000, 1.0],     # upper bounds
    )

    def log_gamma_func(x, A, a, scale):
        val = gamma_func(x, A, a, scale)
        return np.log(np.clip(val, 1e-300, None))

    popt, pcov = curve_fit(
        log_gamma_func, binc[mask], np.log(y[mask]),
        p0=p0, bounds=bounds, maxfev=50000,
    )
    return popt, pcov  # (A, a, scale), covariance


def get_gamma_stats_from_file(filepath, track_group=None):
    """
    Returns records for every centrality bin in the file:
      (energy_GeV, cent_label, mean, sigma, rel_width_pct, rel_width_err_pct,
       alpha, alpha_err, beta, beta_err, track_group)

    alpha is the `a` shape parameter of the gamma fit itself, and beta
    is the `scale` parameter -- both directly comparable to STAR's
    published BES-I alpha/beta values (same fit form).
    rel_width_pct = 100 * sigma / mean is kept as a diagnostic but is
    no longer the primary comparison quantity.
    """
    E = _energy_from_filename(filepath)
    records = []

    with h5py.File(filepath, "r") as f:
        grp = track_group or _detect_track_group(f)
        data_group = f"eventbyeventmeanptcharged/{grp}/centralitybinned"

        g = f[data_group]
        binc = g["bin"][:]            # <pT> bin centers [GeV]
        vals = g["values"][:]         # shape (n_centrality_bins, 1, n_bins)
        cent = f["centrality"][:]     # bin centers [%]
        dcent = f["dcentrality"][:]   # bin half-widths [%]

        for i in range(vals.shape[0]):
            y = vals[i, 0, :]
            c_lo = cent[i] - dcent[i]
            c_hi = cent[i] + dcent[i]
            cent_label = f"{c_lo:.0f}-{c_hi:.0f}%"

            popt, pcov = fit_gamma_to_bin(binc, y)
            A, a, scale = popt

            mean = a * scale
            sigma = np.sqrt(a) * scale
            rel_width_pct = 100.0 * sigma / mean

            # Propagate fit covariance (a, scale) -> Var(sigma/mean) via
            # the delta method. Skip the A row/col of pcov since
            # amplitude doesn't enter sigma/mean. With loc fixed at 0,
            # sigma/mean = 1/sqrt(a) exactly, so this simplifies, but
            # keep the general form in case the model changes later.
            dm_da, dm_dscale = scale, a
            dsg_da, dsg_dscale = 0.5 * scale / np.sqrt(a), np.sqrt(a)

            ds_da = (dsg_da * mean - sigma * dm_da) / mean**2
            ds_dscale = (dsg_dscale * mean - sigma * dm_dscale) / mean**2

            J = np.array([ds_da, ds_dscale])
            cov_sub = pcov[1:3, 1:3]  # rows/cols for (a, scale)
            var_rel_width = J @ cov_sub @ J

            rel_width_err_pct = 100.0 * np.sqrt(max(var_rel_width, 0.0))

            alpha = a
            alpha_err = np.sqrt(max(pcov[1, 1], 0.0))

            beta = scale
            beta_err = np.sqrt(max(pcov[2, 2], 0.0))

            records.append((E, cent_label, mean, sigma, rel_width_pct, rel_width_err_pct,
                             alpha, alpha_err, beta, beta_err, grp))

    return records


In [ ]:
# ------------------------------------------------------------------
# 3. Diagnostic: overlay the raw event-by-event mean-pT histogram
#    with the fitted gamma curve, one panel per energy (0-5% only).
#
# This is the actual check for whether small alpha/beta error bars
# mean a well-constrained fit or whether curve_fit is silently
# struggling (e.g. rank-deficient Jacobian -> inf covariance).
#
# IMPORTANT: y limits are hardcoded to the range where the data
# actually lives (matches eventbyeventmeanptgammafit.ipynb's
# YLIM=(1e-4, 40)). Without this, the fit curve's tails (evaluated
# across the full bin range) fall to ~1e-300, which drags matplotlib's
# log-scale autoscaling down with them and squashes the real data into
# an invisible sliver at the top of the plot -- that's what made the
# fit look bad, not the fit itself.
# ------------------------------------------------------------------
DIAG_XLIM = (0.45, 0.85)
DIAG_YLIM = (1e-4, 40)


def get_single_bin_fit(filepath, cent_label, track_group=None):
    """Returns (E, binc, y, popt, pcov) for the first centrality bin
    in the file matching cent_label."""
    E = _energy_from_filename(filepath)

    with h5py.File(filepath, "r") as f:
        grp = track_group or _detect_track_group(f)
        data_group = f"eventbyeventmeanptcharged/{grp}/centralitybinned"

        g = f[data_group]
        binc = g["bin"][:]
        vals = g["values"][:]
        cent = f["centrality"][:]
        dcent = f["dcentrality"][:]

        for i in range(vals.shape[0]):
            c_lo = cent[i] - dcent[i]
            c_hi = cent[i] + dcent[i]
            if f"{c_lo:.0f}-{c_hi:.0f}%" == cent_label:
                y = vals[i, 0, :]
                popt, pcov = fit_gamma_to_bin(binc, y)
                return E, binc, y, popt, pcov

    raise ValueError(f"No {cent_label} bin found in {filepath}")


def fit_quality_metrics(binc, y, popt):
    """Quantify how well the gamma fit matches the actual data, in the
    same log-space the fit itself was optimized in (mirrors what
    curve_fit was minimizing, so this is a faithful check of fit
    quality -- not a re-judgment using different criteria)."""
    mask = y > 1e-6
    model = gamma_func(binc[mask], *popt)
    log_resid = np.log(y[mask]) - np.log(np.clip(model, 1e-300, None))

    rmse_log = np.sqrt(np.mean(log_resid**2))          # typical log-space miss
    max_abs_log = np.max(np.abs(log_resid))              # worst single-bin miss
    # Convert to human-readable "typical factor off" and "worst factor off"
    typical_factor = np.exp(rmse_log)
    worst_factor = np.exp(max_abs_log)
    return typical_factor, worst_factor


fig, axes = plt.subplots(1, len(filepaths), figsize=(4 * len(filepaths), 4.5), dpi=150, sharey=True)

for ax, fp in zip(axes, filepaths):
    E, binc, y, popt, pcov = get_single_bin_fit(fp, TARGET_CENT_LABEL)
    A, a, scale = popt

    #ax.plot(binc, y, "o", color="tab:blue", markersize=4, alpha=0.7, label="Trajectum data")

    xfit = np.linspace(binc.min(), binc.max(), 1000)
    ax.plot(xfit, gamma_func(xfit, *popt), color="tab:red", lw=2, label="Trajectum fit")

    # STAR overlay: draw the gamma curve using STAR's published alpha/beta
    # at the matched energy, directly on top of the Trajectum data. This
    # visualizes the actual alpha/beta discrepancy in data space, not just
    # as two disconnected numbers in a separate chart.
    #
    # STAR's amplitude isn't published (their curve is a unit-area PDF,
    # ours has a free-fit amplitude A reflecting Trajectum's own histogram
    # normalization) -- so we scale STAR's curve by our OWN fitted A. This
    # is a deliberate choice to make peak heights visually comparable and
    # isolate the SHAPE difference (width/skew), which is what
    # alpha/beta actually control -- amplitude itself isn't a physical
    # quantity being compared here.
    star_E = STAR_ENERGY_MATCH.get(E)
    if star_E is not None and star_E in STAR_ALPHA_BESI and star_E in STAR_BETA_BESI:
        star_a = STAR_ALPHA_BESI[star_E]
        star_b = STAR_BETA_BESI[star_E]
        ax.plot(xfit, A * gamma.pdf(xfit, star_a, loc=0, scale=star_b),
                color="tab:green", linestyle="--", lw=2,
                label=r"STAR BES-I gamma (published $\alpha$,$\beta$)")

    ax.set_yscale("log")
    ax.set_xlim(*DIAG_XLIM)
    ax.set_ylim(*DIAG_YLIM)
    ax.set_xlabel(r"$\langle p_T \rangle$  [GeV]")
    ax.set_title(f"{E:g} GeV", fontsize=13)
    ax.grid(True, alpha=0.25)
    ax.tick_params(axis="both", direction="in")

    alpha_err = np.sqrt(max(pcov[1, 1], 0.0))
    typical_factor, worst_factor = fit_quality_metrics(binc, y, popt)
    label_lines = [rf"$\alpha$={a:.1f}$\pm${alpha_err:.1f}"]
    if star_E is not None and star_E in STAR_ALPHA_BESI:
        label_lines.append(rf"STAR $\alpha$={STAR_ALPHA_BESI[star_E]:.1f}")
    label_lines.append(rf"typical off: {typical_factor:.2f}x")
    label_lines.append(rf"worst off: {worst_factor:.2f}x")
    ax.text(
        0.05, 0.05, "\n".join(label_lines),
        transform=ax.transAxes, fontsize=8, va="bottom", ha="left",
    )

axes[0].set_ylabel(r"P($\langle p_T \rangle$)  [GeV$^{-1}$]")
axes[0].legend(loc="best", fontsize=8)

fig.suptitle("Trajectum vs STAR: event-by-event mean-$p_T$ data + Gamma fits (0-5%)", fontsize=15)
fig.tight_layout()
fig.savefig("graphs/gammafit_data_overlay_0-5.png")
plt.show()
